# S02 — Phase 1 and Phase 2 final selection (VAL only)

Reconciles the retained Phase 1 and Phase 2 model variants on validation data only and exports the paired selection tables.

The public copy is output-stripped; authoritative exported tables and figures are distributed separately in the repository.

In [ ]:
from pathlib import Path
import json
import hashlib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
STAGE_DIR = PROJECT / "Results" / "Final_Article_Harmonized_GEDIAnchored_NaturalP1" / "Temporal_Diagnostics_VAL"
CONTROL_DIR = PROJECT / "Ablations" / "Phase2_LambdaTemporal_Controlled" / "results" / "tables"
OUT = PROJECT / "Results" / "Final_Phase_Selection_VAL_Only"
FIGURES = OUT / "figures"
TABLES = OUT / "tables"
FIGURES.mkdir(parents=True, exist_ok=True)
TABLES.mkdir(parents=True, exist_ok=True)

FOREST_ORDER = ["Ifran", "Maamoura", "Agadir"]
FOREST_COLORS = {"Ifran": "#327DB3", "Maamoura": "#E4B94F", "Agadir": "#D87955"}

# Explicit practical tolerances applied consistently across all study areas.
# These are operational decision thresholds, not statistical significance limits
# and are not described as preregistered or predeclared.
MAX_MAE_DEGRADATION_M = 0.02
MAX_RMSE_DEGRADATION_M = 0.02
MIN_R2_GAIN = 0.002
MIN_SLOPE_GAIN = 0.02
MIN_ABS_D1_REDUCTION_M = 0.005
MIN_ABS_D2_REDUCTION_M = 0.010

print("[VAL-ONLY] No TEST artifact is loaded by this notebook.")
print("Output:", OUT)


In [ ]:
required = {
    "stage_accuracy": STAGE_DIR / "phase1_phase2_gedi_calibration_val.csv",
    "stage_temporal": STAGE_DIR / "phase1_phase2_temporal_summary_val.csv",
    "stage_deltas": STAGE_DIR / "phase2_minus_phase1_temporal_deltas_val.csv",
    "controlled_accuracy": CONTROL_DIR / "03_paired_VAL_accuracy_metrics.csv",
    "controlled_temporal": CONTROL_DIR / "04_paired_VAL_temporal_metrics.csv",
    "controlled_deltas": CONTROL_DIR / "05_lambda_temporal_controlled_deltas.csv",
}
for forest in [x.lower() for x in FOREST_ORDER]:
    required[f"paired_{forest}"] = CONTROL_DIR / f"02_{forest}_paired_val_predictions.csv.gz"
    required[f"zero_raw_{forest}"] = CONTROL_DIR.parent / "cache" / f"{forest}_zero_val_unique_nearest.csv.gz"
    required[f"positive_raw_{forest}"] = CONTROL_DIR.parent / "cache" / f"{forest}_positive_val_unique_nearest.csv.gz"
    required[f"zero_lineage_{forest}"] = CONTROL_DIR.parent / "cache" / f"{forest}_zero_val_unique_nearest.lineage.json"
    required[f"positive_lineage_{forest}"] = CONTROL_DIR.parent / "cache" / f"{forest}_positive_val_unique_nearest.lineage.json"

preflight = pd.DataFrame([
    {"artifact": name, "path": str(path), "exists": path.is_file(),
     "bytes": path.stat().st_size if path.is_file() else 0}
    for name, path in required.items()
])
display(preflight)
if not preflight["exists"].all():
    raise FileNotFoundError("Missing required VAL artifact(s):\n" +
                            preflight.loc[~preflight.exists, "path"].to_string(index=False))
if any("test" in str(path).lower() for path in required.values()):
    raise RuntimeError("A TEST path entered a VAL-only decision notebook.")
preflight.to_csv(TABLES / "00_input_preflight.csv", index=False)
print("[PASS] All inputs exist and all paths are VAL-only.")


In [ ]:
support_rows = []
paired_frames = {}
threeway_frames = {}
for forest in FOREST_ORDER:
    frame = pd.read_csv(required[f"paired_{forest.lower()}"])
    needed = {"aux_shot_uid", "rh95", "pred_zero", "rh95_positive", "pred_positive"}
    missing = needed - set(frame.columns)
    if missing:
        raise KeyError(f"{forest}: missing paired columns {sorted(missing)}")
    frame["aux_shot_uid"] = frame["aux_shot_uid"].astype(str)
    if frame["aux_shot_uid"].duplicated().any():
        raise AssertionError(f"{forest}: duplicate aux_shot_uid in paired VAL file")
    target_equal = np.allclose(frame.rh95, frame.rh95_positive, rtol=0, atol=1e-8, equal_nan=False)
    finite = np.isfinite(frame[["rh95", "pred_zero", "rh95_positive", "pred_positive"]]).all(axis=1)
    if not target_equal or not finite.all():
        raise AssertionError(f"{forest}: paired target mismatch or non-finite values")
    paired_frames[forest] = frame
    zero = pd.read_csv(required[f"zero_raw_{forest.lower()}"])
    positive = pd.read_csv(required[f"positive_raw_{forest.lower()}"])
    for raw in (zero, positive):
        raw["aux_shot_uid"] = raw["aux_shot_uid"].astype(str)
        if raw["aux_shot_uid"].duplicated().any():
            raise AssertionError(f"{forest}: duplicate aux_shot_uid in raw controlled file")
    triple = zero[["aux_shot_uid", "rh95", "pred_off_reference", "pred_on_growthloss"]].merge(
        positive[["aux_shot_uid", "rh95", "pred_on_growthloss"]],
        on="aux_shot_uid", how="inner", validate="one_to_one",
        suffixes=("_lambda0", "_positive"),
    )
    if len(triple) != len(frame) or set(triple.aux_shot_uid) != set(frame.aux_shot_uid):
        raise AssertionError(f"{forest}: three-way support differs from paired support")
    if not np.allclose(triple.rh95_lambda0, triple.rh95_positive, rtol=0, atol=1e-8):
        raise AssertionError(f"{forest}: target mismatch between controlled runs")
    threeway_frames[forest] = triple
    support_rows.append({
        "forest": forest, "n_unique_shots": len(frame),
        "unique_ids": frame.aux_shot_uid.nunique(),
        "targets_identical": target_equal,
        "all_predictions_finite": bool(finite.all()),
    })

support_audit = pd.DataFrame(support_rows)
display(support_audit)
assert (support_audit.n_unique_shots == support_audit.unique_ids).all()
assert support_audit.targets_identical.all() and support_audit.all_predictions_finite.all()
support_audit.to_csv(TABLES / "01_exact_paired_support_audit.csv", index=False)
print("[PASS] lambda=0 and positive-lambda variants use exactly the same unique GEDI IDs and targets.")


In [ ]:
def regression_metrics(y, pred):
    y = np.asarray(y, dtype=float); pred = np.asarray(pred, dtype=float)
    keep = np.isfinite(y) & np.isfinite(pred)
    y, pred = y[keep], pred[keep]
    err = pred - y
    ss_res = float(np.sum(err ** 2))
    ss_tot = float(np.sum((y - y.mean()) ** 2))
    corr = float(np.corrcoef(y, pred)[0, 1])
    slope = float(np.polyfit(y, pred, 1)[0])
    return {
        "n": len(y), "mae": float(np.mean(np.abs(err))),
        "rmse": float(np.sqrt(np.mean(err ** 2))),
        "r2": 1.0 - ss_res / ss_tot, "bias": float(np.mean(err)),
        "corr": corr, "slope": slope,
        "std_ratio": float(np.std(pred, ddof=1) / np.std(y, ddof=1)),
    }

threeway_rows = []
for forest in FOREST_ORDER:
    d = threeway_frames[forest]
    y = d.rh95_lambda0.to_numpy()
    variants = {
        "Phase 1": d.pred_off_reference.to_numpy(),
        "Phase 2 residual only (lambda=0)": d.pred_on_growthloss_lambda0.to_numpy(),
        "Phase 2 temporal": d.pred_on_growthloss_positive.to_numpy(),
    }
    for variant, pred in variants.items():
        threeway_rows.append({"forest": forest, "variant": variant, **regression_metrics(y, pred)})

threeway_metrics = pd.DataFrame(threeway_rows)
display(threeway_metrics)
threeway_metrics.to_csv(TABLES / "02_threeway_exact_paired_val_metrics.csv", index=False)
print("[PASS] Phase 1, residual-only Phase 2, and temporal Phase 2 were recomputed on exact identical GEDI IDs.")


In [ ]:
stage_acc = pd.read_csv(required["stage_accuracy"])
stage_acc["forest"] = stage_acc.forest.str.title()
stage_temp = pd.read_csv(required["stage_temporal"])
stage_temp["forest"] = stage_temp.forest.str.title()
stage_delta = pd.read_csv(required["stage_deltas"])
stage_delta["forest"] = stage_delta.forest.str.title()

expected_phases = {"Phase 1", "Phase 2"}
for forest in FOREST_ORDER:
    block = stage_acc[stage_acc.forest.eq(forest)]
    assert set(block.phase) == expected_phases, (forest, block.phase.tolist())
    assert block.n_gedi.nunique() == 1, f"{forest}: Phase 1 and Phase 2 n differ"
    tblock = stage_temp[stage_temp.forest.eq(forest)]
    assert set(tblock.phase) == expected_phases and tblock.n_crops.nunique() == 1

display(stage_acc.sort_values(["forest", "phase"]))
display(stage_temp.sort_values(["forest", "phase"]))
display(stage_delta.set_index("forest"))
stage_acc.to_csv(TABLES / "02_stage_accuracy_same_val_support.csv", index=False)
stage_temp.to_csv(TABLES / "03_stage_temporal_same_val_crops.csv", index=False)
stage_delta.to_csv(TABLES / "04_stage_deltas_phase2_minus_phase1.csv", index=False)
print("[PASS] Within every forest, Phase 1 and Phase 2 have equal VAL GEDI n and equal crop counts.")


In [ ]:
ctrl_acc = pd.read_csv(required["controlled_accuracy"])
ctrl_temp = pd.read_csv(required["controlled_temporal"])
ctrl_delta = pd.read_csv(required["controlled_deltas"])

display(ctrl_acc.sort_values(["forest", "role"]))
display(ctrl_temp.sort_values(["forest", "role"]))
display(ctrl_delta.set_index("forest"))

ctrl_acc.to_csv(TABLES / "05_controlled_lambda_accuracy.csv", index=False)
ctrl_temp.to_csv(TABLES / "06_controlled_lambda_temporal.csv", index=False)
ctrl_delta.to_csv(TABLES / "07_controlled_lambda_deltas.csv", index=False)


In [ ]:
def sha256_file(path, chunk_size=1024 * 1024):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        while True:
            chunk = stream.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

checkpoint_rows = []
for forest in FOREST_ORDER:
    for role in ("zero", "positive"):
        lineage_path = required[f"{role}_lineage_{forest.lower()}"]
        lineage = json.loads(lineage_path.read_text(encoding="utf-8"))
        checkpoint = Path(lineage["checkpoint"])
        expected_hash = str(lineage.get("checkpoint_sha256", "")).lower()
        exists = checkpoint.is_file()
        actual_hash = sha256_file(checkpoint) if exists else ""
        checkpoint_rows.append({
            "forest": forest,
            "role": role,
            "lambda_interpretation": "residual-only lambda=0" if role == "zero" else "retained positive lambda",
            "checkpoint": str(checkpoint),
            "exists": exists,
            "expected_sha256": expected_hash,
            "actual_sha256": actual_hash,
            "hash_matches": bool(exists and expected_hash and actual_hash.lower() == expected_hash),
            "lineage_test_used": bool(lineage.get("test_used", True)),
        })

checkpoint_audit = pd.DataFrame(checkpoint_rows)
display(checkpoint_audit)
checkpoint_audit.to_csv(TABLES / "08_checkpoint_lineage_and_readiness.csv", index=False)
if checkpoint_audit.lineage_test_used.any():
    raise RuntimeError("A candidate lineage reports TEST use.")
print("[CHECKPOINT AUDIT] Statistical preference and map-production readiness are reported separately.")


In [ ]:
decision_rows = []
for forest in FOREST_ORDER:
    d = stage_delta.loc[stage_delta.forest.eq(forest)].iloc[0]
    p1 = stage_acc[(stage_acc.forest.eq(forest)) & stage_acc.phase.eq("Phase 1")].iloc[0]
    p2 = stage_acc[(stage_acc.forest.eq(forest)) & stage_acc.phase.eq("Phase 2")].iloc[0]
    cd = ctrl_delta.loc[ctrl_delta.forest.eq(forest)].iloc[0]
    tri = threeway_metrics[threeway_metrics.forest.eq(forest)].set_index("variant")
    m_p1 = tri.loc["Phase 1"]
    m_l0 = tri.loc["Phase 2 residual only (lambda=0)"]

    residual_safe = ((m_l0.mae - m_p1.mae) <= MAX_MAE_DEGRADATION_M and
                     (m_l0.rmse - m_p1.rmse) <= MAX_RMSE_DEGRADATION_M)
    residual_benefit = any([
        (m_p1.mae - m_l0.mae) >= 0.01,
        (m_p1.rmse - m_l0.rmse) >= 0.01,
        (m_l0.r2 - m_p1.r2) >= MIN_R2_GAIN,
        (m_l0.slope - m_p1.slope) >= MIN_SLOPE_GAIN,
    ])

    d1_benefit = d.delta_abs_d1_P2_minus_P1 <= -MIN_ABS_D1_REDUCTION_M
    d2_benefit = d.delta_abs_d2_P2_minus_P1 <= -MIN_ABS_D2_REDUCTION_M
    controlled_safe = (cd.delta_mae_positive_minus_zero <= MAX_MAE_DEGRADATION_M and
                       cd.delta_rmse_positive_minus_zero <= MAX_RMSE_DEGRADATION_M)
    controlled_temporal_benefit = any([
        cd.delta_mean_abs_first_difference_m_positive_minus_zero <= -MIN_ABS_D1_REDUCTION_M,
        cd.delta_mean_abs_second_difference_m_positive_minus_zero <= -MIN_ABS_D2_REDUCTION_M,
    ])

    if residual_safe and residual_benefit:
        recommendation = "Phase 2 residual only (lambda=0)"
        if controlled_safe and controlled_temporal_benefit:
            recommendation = "Phase 2 temporal"
    else:
        recommendation = "Phase 1"
    decision_rows.append({
        "forest": forest,
        "n_exact_threeway_val_gedi": int(m_p1.n),
        "n_stage_temporal_val_gedi": int(p1.n_gedi),
        "residual_head_safe_vs_phase1": bool(residual_safe),
        "residual_head_benefit_vs_phase1": bool(residual_benefit),
        "controlled_lambda_safe": bool(controlled_safe),
        "controlled_temporal_benefit": bool(controlled_temporal_benefit),
        "residual_minus_phase1_mae_m": m_l0.mae - m_p1.mae,
        "residual_minus_phase1_rmse_m": m_l0.rmse - m_p1.rmse,
        "residual_minus_phase1_r2": m_l0.r2 - m_p1.r2,
        "residual_minus_phase1_slope": m_l0.slope - m_p1.slope,
        "temporal_minus_residual_mae_m": cd.delta_mae_positive_minus_zero,
        "temporal_minus_residual_rmse_m": cd.delta_rmse_positive_minus_zero,
        "temporal_minus_residual_abs_d1_m": cd.delta_mean_abs_first_difference_m_positive_minus_zero,
        "temporal_minus_residual_abs_d2_m": cd.delta_mean_abs_second_difference_m_positive_minus_zero,
        "stage_phase2_minus_phase1_abs_d1_m": d.delta_abs_d1_P2_minus_P1,
        "stage_phase2_minus_phase1_abs_d2_m": d.delta_abs_d2_P2_minus_P1,
        "recommended_product": recommendation,
    })

decision = pd.DataFrame(decision_rows)
display(decision)
decision.to_csv(TABLES / "09_final_val_only_product_decision.csv", index=False)


In [ ]:
plot = stage_delta.set_index("forest").loc[FOREST_ORDER]
colors = [FOREST_COLORS[f] for f in FOREST_ORDER]

fig, axes = plt.subplots(1, 3, figsize=(12.5, 3.6), dpi=150)
specs = [
    ("delta_mae_P2_minus_P1", "Delta MAE (m)", "Lower favours Phase 2"),
    ("delta_rmse_P2_minus_P1", "Delta RMSE (m)", "Lower favours Phase 2"),
    ("delta_r2_P2_minus_P1", "Delta R2", "Higher favours Phase 2"),
]
for ax, (col, ylabel, subtitle) in zip(axes, specs):
    vals = plot[col].to_numpy()
    ax.bar(FOREST_ORDER, vals, color=colors, edgecolor="0.25", linewidth=0.7)
    ax.axhline(0, color="0.15", linestyle="--", linewidth=0.9)
    ax.set_ylabel(ylabel)
    ax.set_title(subtitle, fontsize=10)
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.22)
fig.suptitle("Phase 2 minus Phase 1 on identical validation support", fontweight="bold")
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(FIGURES / f"phase2_minus_phase1_val_accuracy.{ext}", dpi=300, bbox_inches="tight")
plt.show(); plt.close(fig)

fig, axes = plt.subplots(1, 2, figsize=(9.4, 3.8), dpi=150)
for ax, col, title in [
    (axes[0], "delta_abs_d1_P2_minus_P1", "Mean absolute first difference"),
    (axes[1], "delta_abs_d2_P2_minus_P1", "Mean absolute second difference"),
]:
    vals = plot[col].to_numpy()
    ax.bar(FOREST_ORDER, vals, color=colors, edgecolor="0.25", linewidth=0.7)
    ax.axhline(0, color="0.15", linestyle="--", linewidth=0.9)
    ax.set_ylabel("Phase 2 - Phase 1 (m)")
    ax.set_title(title)
    ax.tick_params(axis="x", rotation=20)
    ax.grid(axis="y", alpha=0.22)
fig.suptitle("Temporal-coherence change on identical validation crops\nNegative values indicate smoother Phase 2 trajectories", fontweight="bold")
fig.tight_layout()
for ext in ("png", "pdf"):
    fig.savefig(FIGURES / f"phase2_minus_phase1_val_temporal.{ext}", dpi=300, bbox_inches="tight")
plt.show(); plt.close(fig)


In [ ]:
recommendations = dict(zip(decision.forest, decision.recommended_product))
phase1_sites = [f for f in FOREST_ORDER if recommendations[f] == "Phase 1"]
residual_sites = [f for f in FOREST_ORDER if "residual only" in recommendations[f]]
temporal_sites = [f for f in FOREST_ORDER if recommendations[f] == "Phase 2 temporal"]

paragraph = (
    "Phase 1 and Phase 2 were compared on identical spatial-validation support within each study area. "
    "Phase 2 was retained only when it improved temporal coherence or calibration without a material loss "
    "of GEDI-supported accuracy. Under explicit practical tolerances applied consistently across study areas, "
    + (f"Phase 1 was retained for {', '.join(phase1_sites)}, " if phase1_sites else "")
    + (f"the residual-only Phase 2 head was retained for {', '.join(residual_sites)}, " if residual_sites else "")
    + (f"and temporal Phase 2 was retained for {', '.join(temporal_sites)}." if temporal_sites else "")
    + " The controlled lambda_temp=0 comparison was used to distinguish the residual-head contribution "
      "from that of the temporal loss."
)
print(paragraph)
(OUT / "manuscript_ready_val_selection_paragraph.txt").write_text(paragraph + "\n", encoding="utf-8")

actions = []
for forest in FOREST_ORDER:
    product = recommendations[forest]
    required_role = "positive" if product == "Phase 2 temporal" else ("zero" if "residual only" in product else None)
    if required_role is None:
        checkpoint_ready = False
        checkpoint_path = "Phase 1 checkpoint must be audited in the Phase 1 lineage"
        checkpoint_hash = ""
    else:
        row = checkpoint_audit[(checkpoint_audit.forest.eq(forest)) & checkpoint_audit.role.eq(required_role)].iloc[0]
        checkpoint_ready = bool(row.exists and row.hash_matches and not row.lineage_test_used)
        checkpoint_path = row.checkpoint
        checkpoint_hash = row.actual_sha256
    actions.append({
        "forest": forest,
        "final_product": product,
        "required_checkpoint_role": required_role or "phase1",
        "checkpoint_ready": checkpoint_ready,
        "checkpoint": checkpoint_path,
        "checkpoint_sha256": checkpoint_hash,
        "dense_inference_action": (
            "AUDIT PHASE 1 CHECKPOINT BEFORE INFERENCE"
            if product == "Phase 1" else
            (f"READY: regenerate dense maps and downstream products from {product}"
             if checkpoint_ready else f"BLOCKED: recover and verify the checkpoint for {product}")
        )
    })
actions = pd.DataFrame(actions)
display(actions)
actions.to_csv(TABLES / "10_inference_and_reporting_actions.csv", index=False)

manifest = {
    "selection_partition": "spatial validation only",
    "test_used_for_selection": False,
    "decision_rule_status": "explicit practical sensitivity rule; not preregistered",
    "thresholds": {
        "max_mae_degradation_m": MAX_MAE_DEGRADATION_M,
        "max_rmse_degradation_m": MAX_RMSE_DEGRADATION_M,
        "min_r2_gain": MIN_R2_GAIN,
        "min_slope_gain": MIN_SLOPE_GAIN,
        "min_abs_d1_reduction_m": MIN_ABS_D1_REDUCTION_M,
        "min_abs_d2_reduction_m": MIN_ABS_D2_REDUCTION_M,
    },
    "recommendations": recommendations,
    "inputs": {k: str(v) for k, v in required.items()},
}
(OUT / "selection_manifest.json").write_text(json.dumps(manifest, indent=2), encoding="utf-8")
print("[DONE] Decision tables, figures, manuscript wording and inference actions saved under", OUT)
